In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import pandas as pd

# Load datasets
inpatient = pd.read_csv("/content/Train_Inpatientdata-1542865627584.csv", on_bad_lines='skip')
outpatient = pd.read_csv("/content/Train_Outpatientdata-1542865627584.csv")
beneficiary = pd.read_csv("/content/Train_Beneficiarydata-1542865627584.csv")
provider_labels = pd.read_csv("/content/Train-1542865627584.csv")   # FRAUD labels

# Add claim type
inpatient["claim_type"] = "inpatient"
outpatient["claim_type"] = "outpatient"

# Combine inpatient + outpatient
claims = pd.concat([inpatient, outpatient], ignore_index=True)

# Merge beneficiary (patient info)
claims = claims.merge(beneficiary, on="BeneID", how="left")

# Merge fraud labels (hospital info)
claims = claims.merge(provider_labels, on="Provider", how="left")   # FIXED HERE

# Convert Yes/No to 0/1
claims["PotentialFraud"] = (claims["PotentialFraud"] == "Yes").astype(int)

In [ ]:
claims.head()
claims.shape
claims.isnull().sum()

,0
BeneID,0
ClaimID,0
ClaimStartDt,0
ClaimEndDt,0
Provider,0
InscClaimAmtReimbursed,0
AttendingPhysician,1508
OperatingPhysician,443764
OtherPhysician,358475
AdmissionDt,517737


In [ ]:
claims['ClaimStartDt'] = pd.to_datetime(claims['ClaimStartDt'], errors="coerce")
claims['ClaimEndDt'] = pd.to_datetime(claims['ClaimEndDt'], errors="coerce")
claims['AdmissionDt'] = pd.to_datetime(claims['AdmissionDt'], errors="coerce")
claims['DischargeDt'] = pd.to_datetime(claims['DischargeDt'], errors="coerce")
claims['DOB'] = pd.to_datetime(claims['DOB'], errors="coerce")

In [ ]:
claims[['ClaimStartDt','ClaimEndDt', 'AdmissionDt', 'DischargeDt', 'DOB']].head()

,ClaimStartDt,ClaimEndDt,AdmissionDt,DischargeDt,DOB
0,2009-04-12,2009-04-18,2009-04-12,2009-04-18,1943-01-01
1,2009-08-31,2009-09-02,2009-08-31,2009-09-02,1943-01-01
2,2009-09-17,2009-09-20,2009-09-17,2009-09-20,1943-01-01
3,2009-02-14,2009-02-22,2009-02-14,2009-02-22,1914-03-01
4,2009-08-13,2009-08-30,2009-08-13,2009-08-30,1938-04-01


In [ ]:
claims['claimDuration'] = (claims['ClaimEndDt'] - claims['ClaimStartDt']).dt.days

In [ ]:
claims['HospitalStayDays'] = (claims['DischargeDt'] - claims['AdmissionDt']).dt.days

In [ ]:
claims['PatientAge'] = (pd.Timestamp('2020-01-01') - claims['DOB']).dt.days // 365

In [ ]:
claims[['claimDuration', 'HospitalStayDays', 'PatientAge']].head()

,claimDuration,HospitalStayDays,PatientAge
0,6,6.0,77
1,2,2.0,77
2,3,3.0,77
3,8,8.0,105
4,17,17.0,81


In [ ]:
claims['PotentialFraud'].value_counts()


,count
PotentialFraud,
0,345415
1,212796


In [ ]:
claims['PotentialFraud'].value_counts(normalize=True)*100

,proportion
PotentialFraud,
0,61.878931
1,38.121069


In [ ]:
claims.groupby("PotentialFraud")["InscClaimAmtReimbursed"].describe()

,count,mean,std,min,25%,50%,75%,max
PotentialFraud,,,,,,,,
0,345415.0,755.213352,3056.460166,0.0,40.0,80.0,300.0,125000.0
1,212796.0,1389.505066,4785.074685,0.0,40.0,90.0,400.0,125000.0


In [ ]:
claims.groupby("PotentialFraud")['PotentialFraud'].describe()

,count,mean,std,min,25%,50%,75%,max
PotentialFraud,,,,,,,,
0,345415.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,212796.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0


In [ ]:
claims.groupby("PotentialFraud")['claimDuration'].describe()

,count,mean,std,min,25%,50%,75%,max
PotentialFraud,,,,,,,,
0,345415.0,1.617680,4.813118,0.0,0.0,0.0,0.0,35.0
1,212796.0,1.906916,5.045431,0.0,0.0,0.0,0.0,36.0


In [ ]:
claims.groupby("PotentialFraud")['DiagnosisGroupCode'].value_counts().head(10)

PotentialFraud  DiagnosisGroupCode
0               884                   77
                167                   76
                183                   76
                208                   75
                881                   70
                940                   70
                187                   69
                887                   69
                941                   69
                882                   68
Name: count, dtype: int64

In [ ]:
claims.groupby("PotentialFraud")['DiagnosisGroupCode'].value_counts().head(20)

PotentialFraud  DiagnosisGroupCode
0               884                   77
                167                   76
                183                   76
                208                   75
                881                   70
                940                   70
                187                   69
                887                   69
                941                   69
                882                   68
                182                   67
                939                   67
                168                   66
                876                   66
                184                   65
                205                   65
                206                   65
                177                   64
                180                   64
                204                   64
Name: count, dtype: int64

In [ ]:
claims.groupby("PotentialFraud")['ClmProcedureCode_1'].value_counts().head(15)

PotentialFraud  ClmProcedureCode_1
0               9904.0                518
                8154.0                407
                3893.0                371
                66.0                  362
                3995.0                332
                4516.0                281
                3722.0                255
                8151.0                182
                8872.0                167
                9671.0                159
                4513.0                146
                5123.0                132
                9390.0                125
                9672.0                120
                7935.0                117
Name: count, dtype: int64

In [ ]:
claims.groupby("PotentialFraud")['ClmProcedureCode_1'].value_counts().head(20)

PotentialFraud  ClmProcedureCode_1
0               9904.0                518
                8154.0                407
                3893.0                371
                66.0                  362
                3995.0                332
                4516.0                281
                3722.0                255
                8151.0                182
                8872.0                167
                9671.0                159
                4513.0                146
                5123.0                132
                9390.0                125
                9672.0                120
                7935.0                117
                8152.0                111
                9339.0                110
                3491.0                 94
                3772.0                 90
                3812.0                 86
Name: count, dtype: int64

In [ ]:
claims.groupby("PotentialFraud")['AttendingPhysician'].value_counts().head(10)

PotentialFraud  AttendingPhysician
0               PHY351121             1053
                PHY375943              912
                PHY432614              716
                PHY326984              686
                PHY362889              674
                PHY389456              673
                PHY367255              634
                PHY373032              618
                PHY356444              600
                PHY360179              583
Name: count, dtype: int64

In [ ]:
claims.groupby("PotentialFraud")['AttendingPhysician'].value_counts().head(20)

PotentialFraud  AttendingPhysician
0               PHY351121             1053
                PHY375943              912
                PHY432614              716
                PHY326984              686
                PHY362889              674
                PHY389456              673
                PHY367255              634
                PHY373032              618
                PHY356444              600
                PHY360179              583
                PHY405720              544
                PHY387900              514
                PHY430054              513
                PHY342223              503
                PHY361063              470
                PHY388040              447
                PHY326049              445
                PHY403755              429
                PHY351973              426
                PHY318242              422
Name: count, dtype: int64

In [ ]:
claims['InscClaimAmtReimbursed'] = pd.to_numeric(claims['InscClaimAmtReimbursed'], errors='coerce')

provider_summary = claims.groupby('Provider').agg(
    total_claims=('ClaimID', 'count'),
    unique_doctors=('AttendingPhysician', 'nunique'),
    avg_claim_amount=('InscClaimAmtReimbursed', 'mean'),
    fraud_ratio=('PotentialFraud', 'mean')
).reset_index()

provider_summary.head()

,Provider,total_claims,unique_doctors,avg_claim_amount,fraud_ratio
0,PRV51001,25,14,4185.600000,0.0
1,PRV51003,132,44,4588.409091,1.0
2,PRV51004,149,38,350.134228,0.0
3,PRV51005,1165,6,241.124464,1.0
4,PRV51007,72,10,468.194444,0.0


In [ ]:
claims = claims.merge(provider_summary, on="Provider", how="left")
claims.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,...,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt,PotentialFraud,claimDuration,HospitalStayDays,PatientAge,total_claims,unique_doctors,avg_claim_amount,fraud_ratio
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,...,60,70,1,6,6.0,77,107,68,7010.093458,1.0
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,...,60,70,0,2,2.0,77,243,81,2461.646091,0.0
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,...,60,70,0,3,3.0,77,20,5,6084.000000,0.0
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,...,250,320,0,8,8.0,105,89,17,2124.382022,0.0
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,...,120,100,0,17,17.0,81,26,14,4850.000000,0.0


In [ ]:
claims['Gender'] = claims['Gender'].map({"M": 1, "F": 0})

In [ ]:
claims['Gender'].value_counts()

,count
Gender,


In [ ]:
race_dummies = pd.get_dummies(claims['Race'], prefix='Race')
claims = pd.concat([claims, race_dummies], axis=1)

In [ ]:
print(claims.columns.tolist())

['BeneID', 'ClaimID', 'ClaimStartDt', 'ClaimEndDt', 'Provider', 'InscClaimAmtReimbursed', 'AttendingPhysician', 'OperatingPhysician', 'OtherPhysician', 'AdmissionDt', 'ClmAdmitDiagnosisCode', 'DeductibleAmtPaid', 'DischargeDt', 'DiagnosisGroupCode', 'ClmDiagnosisCode_1', 'ClmDiagnosisCode_2', 'ClmDiagnosisCode_3', 'ClmDiagnosisCode_4', 'ClmDiagnosisCode_5', 'ClmDiagnosisCode_6', 'ClmDiagnosisCode_7', 'ClmDiagnosisCode_8', 'ClmDiagnosisCode_9', 'ClmDiagnosisCode_10', 'ClmProcedureCode_1', 'ClmProcedureCode_2', 'ClmProcedureCode_3', 'ClmProcedureCode_4', 'ClmProcedureCode_5', 'ClmProcedureCode_6', 'claim_type', 'DOB', 'DOD', 'Gender', 'Race', 'RenalDiseaseIndicator', 'State', 'County', 'NoOfMonths_PartACov', 'NoOfMonths_PartBCov', 'ChronicCond_Alzheimer', 'ChronicCond_Heartfailure', 'ChronicCond_KidneyDisease', 'ChronicCond_Cancer', 'ChronicCond_ObstrPulmonary', 'ChronicCond_Depression', 'ChronicCond_Diabetes', 'ChronicCond_IschemicHeart', 'ChronicCond_Osteoporasis', 'ChronicCond_rheumat

In [ ]:
claims['Gender'].head(20)

,Gender
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN
5,NaN
6,NaN
7,NaN
8,NaN
9,NaN


In [ ]:
beneficiary['Gender'].value_counts(dropna=False).head()

,count
Gender,
2,79106
1,59450


In [ ]:
claims = claims.merge(beneficiary, on="BeneID", how='left')

In [ ]:
claims['BeneID'].isin(beneficiary['BeneID']).mean()*100

np.float64(100.0)

In [ ]:
claims = pd.merge(
    claims,
    beneficiary[["BeneID", "Gender", "Race"]],
    how="left",
    on="BeneID"
)

In [ ]:
claims['Gender'] = claims['Gender'].map({1: 1, 2: 0})

In [ ]:
claims['Gender'].value_counts(dropna=False)

,count
Gender,
0,323114
1,235097


In [ ]:
claims['Race'].value_counts(dropna=False)

,count
Race,
1,471036
2,55640
3,19715
5,11820


In [ ]:
race_dummies = pd.get_dummies(claims['Race'], prefix='Race')
claims = pd.concat([claims, race_dummies], axis=1)


In [ ]:
race_dummies.head()

,Race_1,Race_2,Race_3,Race_5
0,True,False,False,False
1,True,False,False,False
2,True,False,False,False
3,False,True,False,False
4,True,False,False,False


In [ ]:
for col in race_dummies.columns:
  claims[col] = claims[col].astype(int)

In [ ]:
claims[['Race_1', 'Race_2', 'Race_3', 'Race_5']].head()

,Race_1,Race_1,Race_2,Race_2,Race_3,Race_3,Race_5,Race_5
0,1,1,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0
3,0,0,1,1,0,0,0,0
4,1,1,0,0,0,0,0,0


In [ ]:
# List of columns to drop explicitly, beyond those with _x or _y suffixes
explicit_drop_cols = [
    'ClaimID',
    'ClaimStartDt',
    'ClaimEndDt',
    'AdmissionDt',
    'DischargeDt'
]

# Identify columns to drop due to merging suffixes and redundant info
suffix_drop_cols = [col for col in claims.columns if col.endswith('_x') or col.endswith('_y')]

# Combine lists and remove duplicates
drop_cols = list(set(explicit_drop_cols + suffix_drop_cols))

# Drop the columns, using errors='ignore' to prevent error if a column is already missing
claims = claims.drop(columns=drop_cols, errors='ignore')

In [ ]:
diag_cols = [col for col in claims.columns if "ClmAdmitDiagnosisCode" in col]
claims['num_diagnosis_codes'] = claims[diag_cols].notnull().sum(axis=1)

In [ ]:
proc_cols = [col for col in claims.columns if "ClamProcedureCode" in col]
claims['num_procedure_codes'] = claims[proc_cols].notnull().sum(axis=1)

In [ ]:
chronic_cols = [col for col in claims.columns if "ChronicCond_" in col]
claims['chornic_count'] = claims[chronic_cols].sum(axis=1)

In [ ]:
claims.to_csv("clean_claims_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("clean_claims_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>